In [1]:
import numpy as np
import os

snapshots = [99, 50, 33, 25]

fields = [
    "SubhaloMassType",
    "SubhaloSFR",
    "SubhaloGasMetallicitySfrWeighted"
]

for snap in snapshots:

    filename = f"TNG_groupcat_data/snapshot_{snap}.npz"

    print("\n" + "=" * 60)
    print(f"SNAPSHOT {snap}")
    print("=" * 60)

    if not os.path.exists(filename):
        print("Fajl ne postoji!")
        continue

    data = np.load(filename)

    # Broj ID-jeva
    ids = data["SubhaloID"]

    print(f"\nBroj subhaloa (ID): {len(ids)}")

    # Provera svakog polja
    lengths = []

    for field in fields:

        array = data[field]

        lengths.append(len(array))

        print(f"{field}:")
        print(f"    Oblik: {array.shape}")
        print(f"    Broj redova: {len(array)}")

    # Provera jednakog broja redova
    if len(set(lengths + [len(ids)])) == 1:

        print("\nSVI NIZOVI IMAJU ISTI BROJ REDOVA.")

    else:

        print("\nGRESKA: BROJ REDOVA SE NE POKLAPA")

    # Provera ID-jeva
    expected_ids = np.arange(len(ids))

    if np.array_equal(ids, expected_ids):

        print("ID-jevi odgovaraju indeksima u katalogu.")

    else:

        print("GRESKA: ID-jevi nisu pravilno poređani")

    # Provera da li postoje duplikati
    if len(np.unique(ids)) == len(ids):

        print("Nema dupliranih ID-jeva.")

    else:

        print("GRESKA: Postoje duplirani ID-jevi.")

    # Provera nedostajućih vrednosti
    for field in fields:

        array = data[field]

        print(
            f"{field} | NaN vrednosti:",
            np.isnan(array).sum()
        )

print("\nKraj")


SNAPSHOT 99

Broj subhaloa (ID): 4371211
SubhaloMassType:
    Oblik: (4371211, 6)
    Broj redova: 4371211
SubhaloSFR:
    Oblik: (4371211,)
    Broj redova: 4371211
SubhaloGasMetallicitySfrWeighted:
    Oblik: (4371211,)
    Broj redova: 4371211

SVI NIZOVI IMAJU ISTI BROJ REDOVA.
ID-jevi odgovaraju indeksima u katalogu.
Nema dupliranih ID-jeva.
SubhaloMassType | NaN vrednosti: 0
SubhaloSFR | NaN vrednosti: 0
SubhaloGasMetallicitySfrWeighted | NaN vrednosti: 0

SNAPSHOT 50

Broj subhaloa (ID): 5127294
SubhaloMassType:
    Oblik: (5127294, 6)
    Broj redova: 5127294
SubhaloSFR:
    Oblik: (5127294,)
    Broj redova: 5127294
SubhaloGasMetallicitySfrWeighted:
    Oblik: (5127294,)
    Broj redova: 5127294

SVI NIZOVI IMAJU ISTI BROJ REDOVA.
ID-jevi odgovaraju indeksima u katalogu.
Nema dupliranih ID-jeva.
SubhaloMassType | NaN vrednosti: 0
SubhaloSFR | NaN vrednosti: 0
SubhaloGasMetallicitySfrWeighted | NaN vrednosti: 0

SNAPSHOT 33

Broj subhaloa (ID): 5621338
SubhaloMassType:
    Obl

In [2]:
import numpy as np

snapshots = [99, 50, 33, 25]

all_data = {}

for snap in snapshots:

    filename = f"TNG_groupcat_data/snapshot_{snap}.npz"

    loaded = np.load(filename)

    all_data[snap] = {
        key: loaded[key]
        for key in loaded.files
    }

    loaded.close()

    print(f'Snapshot {snap}: učitan')

Snapshot 99: učitan
Snapshot 50: učitan
Snapshot 33: učitan
Snapshot 25: učitan


In [3]:
snapshots = [99, 50, 33, 25]

filtered_data = {}

for snap in snapshots:

    print("\n" + "=" * 50)
    print(f"SNAPSHOT {snap}")
    print("=" * 50)

    data = all_data[snap]

    # Ukupan broj objekata pre filtriranja
    N_before = len(data["SubhaloID"])

    # Zvezdana masa (indeks 4)
    stellar_mass = data["SubhaloMassType"][:, 4]

    # Selekcija objekata sa pozitivnom zvezdanom masom
    mask = stellar_mass > 0

    # Broj objekata nakon filtriranja
    N_after = np.sum(mask)

    # Broj izbačenih objekata
    N_removed = N_before - N_after

    # Filtriranje svih polja istom maskom
    filtered_data[snap] = {
        field: data[field][mask]
        for field in [
            "SubhaloID",
            "SubhaloMassType",
            "SubhaloSFR",
            "SubhaloGasMetallicitySfrWeighted"
        ]
    }

    print("Broj objekata pre filtriranja:", N_before)
    print("Broj objekata nakon filtriranja:", N_after)
    print("Broj izbačenih objekata:", N_removed)

    print(
        "Procenat izbačenih objekata:",
        round(100 * N_removed / N_before, 2), "%"
    )

print("\nKraj")


SNAPSHOT 99
Broj objekata pre filtriranja: 4371211
Broj objekata nakon filtriranja: 337261
Broj izbačenih objekata: 4033950
Procenat izbačenih objekata: 92.28 %

SNAPSHOT 50
Broj objekata pre filtriranja: 5127294
Broj objekata nakon filtriranja: 492655
Broj izbačenih objekata: 4634639
Procenat izbačenih objekata: 90.39 %

SNAPSHOT 33
Broj objekata pre filtriranja: 5621338
Broj objekata nakon filtriranja: 680244
Broj izbačenih objekata: 4941094
Procenat izbačenih objekata: 87.9 %

SNAPSHOT 25
Broj objekata pre filtriranja: 5828784
Broj objekata nakon filtriranja: 836537
Broj izbačenih objekata: 4992247
Procenat izbačenih objekata: 85.65 %

Kraj


# 0.2 dex maseni prozor

In [8]:
protoMWmass = [10.68, 10.48, 10.28, 9.98]

snapshot_redshifts = {
    99: 0,
    50: 1,
    33: 2,
    25: 3
}

h = 0.6774

mass_window_counts = {}

galaxies_in_mass_window = {}

for snap, z in snapshot_redshifts.items():

    data = filtered_data[snap]

    stellar_mass = data["SubhaloMassType"][:, 4]

    mass_msun = stellar_mass * 1e10 / h

    logM = np.full(len(mass_msun), np.nan)

    valid_mass = (
        np.isfinite(mass_msun) &
        (mass_msun > 0)
    )

    logM[valid_mass] = np.log10(mass_msun[valid_mass])

    MW_mass = protoMWmass[z]

    lower_limit = MW_mass - 0.2
    upper_limit = MW_mass + 0.2

    mass_mask = (
        valid_mass &
        (logM >= lower_limit) &
        (logM <= upper_limit)
    )

    # SFR galaksija unutar masenog prozora
    sfr_mass_window = data["SubhaloSFR"][mass_mask]

    # Galaksije sa SFR = 0
    zero_sfr = sfr_mass_window == 0
    n_zero_sfr = np.sum(zero_sfr)

    # Galaksije sa logSFR < -4
    positive_sfr = sfr_mass_window[sfr_mass_window > 0]
    logSFR = np.log10(positive_sfr)
    n_low_sfr = np.sum(logSFR < -4)

    galaxies_in_mass_window[z] = data["SubhaloID"][mass_mask]
    mass_window_counts[z] = np.sum(mass_mask)

    print(f"SNAPSHOT {snap} | z = {z}")
    print(f"Broj galaksija unutar masenog prozora: {mass_window_counts[z]}")
    
    print(f"Broj galaksija sa SFR = 0: {n_zero_sfr}")
    print(f"Broj galaksija sa logSFR < -4: {n_low_sfr}")

SNAPSHOT 99 | z = 0
Broj galaksija unutar masenog prozora: 1972
Broj galaksija sa SFR = 0: 790
Broj galaksija sa logSFR < -4: 0
SNAPSHOT 50 | z = 1
Broj galaksija unutar masenog prozora: 1930
Broj galaksija sa SFR = 0: 108
Broj galaksija sa logSFR < -4: 0
SNAPSHOT 33 | z = 2
Broj galaksija unutar masenog prozora: 1492
Broj galaksija sa SFR = 0: 6
Broj galaksija sa logSFR < -4: 0
SNAPSHOT 25 | z = 3
Broj galaksija unutar masenog prozora: 1115
Broj galaksija sa SFR = 0: 2
Broj galaksija sa logSFR < -4: 0


# Provera minimalnog SFR

In [9]:
for snap, z in snapshot_redshifts.items():

    data = filtered_data[snap]

    stellar_mass = data["SubhaloMassType"][:, 4]
    mass_msun = stellar_mass * 1e10 / h

    valid_mass = np.isfinite(mass_msun) & (mass_msun > 0)
    logM = np.full(len(mass_msun), np.nan)
    logM[valid_mass] = np.log10(mass_msun[valid_mass])

    MW_mass = protoMWmass[z]

    mass_mask = (
        valid_mass &
        (logM >= MW_mass - 0.2) &
        (logM <= MW_mass + 0.2)
    )

    sfr = data["SubhaloSFR"][mass_mask]
    positive_sfr = sfr[sfr > 0]

    print(f"\nz = {z}")
    print(f"Minimum positive SFR: {positive_sfr.min():.6e}")
    print(f"Maximum SFR: {sfr.max():.6e}")
    print(f"Minimum logSFR: {np.log10(positive_sfr).min():.4f}")
    print(f"Broj pozitivnih SFR: {len(positive_sfr)}")


z = 0
Minimum positive SFR: 3.112666e-04
Maximum SFR: 3.628362e+01
Minimum logSFR: -3.5069
Broj pozitivnih SFR: 1182

z = 1
Minimum positive SFR: 6.660243e-04
Maximum SFR: 1.164635e+02
Minimum logSFR: -3.1765
Broj pozitivnih SFR: 1822

z = 2
Minimum positive SFR: 3.266373e-03
Maximum SFR: 1.078789e+02
Minimum logSFR: -2.4859
Broj pozitivnih SFR: 1486

z = 3
Minimum positive SFR: 5.036292e-01
Maximum SFR: 1.354088e+02
Minimum logSFR: -0.2979
Broj pozitivnih SFR: 1113
